# Dataset Prep

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

### Load Dataset

In [2]:
hurricanes = pd.read_csv("data/cleaned_hurricanes.csv") # renamed kaggle csv from storms to hurricane_data
print(hurricanes.columns.tolist())

['name', 'year', 'month', 'day', 'hour', 'lat', 'long', 'status', 'category', 'wind', 'pressure', 'tropicalstorm_force_diameter', 'hurricane_force_diameter', 'hurricane_class', 'merge_name', 'storm_id', 'correct_datetime', 'datetime', 'CSST', 'COHC', 'SHRD', 'RHMD', 'MTPW', 'VMPI', 'DTL']


# Feature Selection

### Logistic Regression Helper Function

In [3]:
def fit_logistic_model(X, y):
    model = sm.Logit(y, X)
    result = model.fit(method="bfgs", disp=False, maxiter=100)
    return result

### Forward Selection (AIC)

In [4]:
def forward_selection_aic(X, y):

    if "const" not in X.columns:
        X = sm.add_constant(X)

    remaining = list(X.columns)
    remaining.remove("const")

    selected = ["const"]
    current_aic = np.inf

    while remaining:

        scores = []

        for variable in remaining:
            candidate_vars = selected + [variable]
            result = fit_logistic_model(
                X[candidate_vars],
                y
            )

            scores.append(
                (result.aic, variable)
            )

        best_aic, best_variable = min(scores)

        if best_aic < current_aic:
            selected.append(best_variable)
            remaining.remove(best_variable)
            current_aic = best_aic

        else:
            break

    final_model = fit_logistic_model(
        X[selected],
        y
    )

    return final_model, selected

### Backward Selection (AIC)

In [5]:
def backward_selection_aic(X, y):

    selected = list(X.columns)

    current_model = fit_logistic_model(
        X[selected],
        y
    )

    current_aic = current_model.aic

    while len(selected) > 1:

        scores = []

        for variable in selected:
            if variable == "const":
                continue

            candidate_vars = selected.copy()
            candidate_vars.remove(variable)

            result = fit_logistic_model(
                X[candidate_vars],
                y
            )

            scores.append(
                (result.aic, variable)
            )

        best_aic, variable_to_remove = min(scores)

        if best_aic < current_aic:
            selected.remove(variable_to_remove)
            current_aic = best_aic

        else:
            break

    final_model = fit_logistic_model(
        X[selected],
        y
    )

    return final_model, selected

### Stepwise Selection (AIC)

In [6]:
def stepwise_selection_aic(X, y):

    # Work with a copy so the original X is not changed
    X_model = X.copy()

    # Add the intercept column if it is missing
    if "const" not in X_model.columns:
        X_model = sm.add_constant(X_model)

    remaining = [
        variable
        for variable in X_model.columns
        if variable != "const"
    ]

    selected = ["const"]
    current_aic = np.inf
    changed = True

    while changed:
        changed = False

        # Forward step
        forward_scores = []

        for variable in remaining:
            candidate_vars = selected + [variable]

            result = fit_logistic_model(
                X_model[candidate_vars],
                y
            )

            forward_scores.append(
                (result.aic, variable)
            )

        if forward_scores:
            best_aic, best_variable = min(forward_scores)

            if best_aic < current_aic:
                selected.append(best_variable)
                remaining.remove(best_variable)
                current_aic = best_aic
                changed = True

        # Backward step
        backward_scores = []

        for variable in selected:
            if variable == "const":
                continue

            candidate_vars = selected.copy()
            candidate_vars.remove(variable)

            result = fit_logistic_model(
                X_model[candidate_vars],
                y
            )

            backward_scores.append(
                (result.aic, variable)
            )

        if backward_scores:
            best_aic, variable_to_remove = min(backward_scores)

            if best_aic < current_aic:
                selected.remove(variable_to_remove)
                remaining.append(variable_to_remove)
                current_aic = best_aic
                changed = True

    final_model = fit_logistic_model(
        X_model[selected],
        y
    )

    return final_model, selected

### Model 1: Operational Model

Given everything we know about the storm (except wind), what predicts a major hurricane?

In [7]:
operational_predictors = [
    "year",
    "month",
    "day",
    "hour",
    "lat",
    "long",
    "tropicalstorm_force_diameter",
    "hurricane_force_diameter",
    "CSST",
    "COHC",
    "SHRD",
    "RHMD",
    "MTPW",
    "VMPI",
    "DTL"
]

target = "hurricane_class"

model1_data = hurricanes[operational_predictors + [target]].dropna()

X1 = model1_data[operational_predictors]
y1 = model1_data[target]

X1 = sm.add_constant(X1, has_constant="add")

In [8]:
forward1_model, forward1_vars = forward_selection_aic(X1, y1)
backward1_model, backward1_vars = backward_selection_aic(X1, y1)
stepwise1_model, stepwise1_vars = stepwise_selection_aic(X1, y1)

/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analy

### Model 2: Environmental Model

Can environmental conditions alone distinguish storms that become major hurricanes?

In [9]:
environmental_predictors = [
    "year",
    "month",
    "day",
    "hour",
    "lat",
    "long",
    "CSST",
    "COHC",
    "SHRD",
    "RHMD",
    "MTPW",
    "VMPI",
    "DTL"
]

target = "hurricane_class"

model2_data = hurricanes[environmental_predictors + [target]].dropna()

X2 = model2_data[environmental_predictors]
y2 = model2_data[target]

X2 = sm.add_constant(X2, has_constant="add")

In [10]:
forward2_model, forward2_vars = forward_selection_aic(X2, y2)
backward2_model, backward2_vars = backward_selection_aic(X2, y2)
stepwise2_model, stepwise2_vars = stepwise_selection_aic(X2, y2)

/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/DMII_Group_Project/hurricane-analy

### Comparison table for Model 1

In [11]:
operational_results = pd.DataFrame({
    "Method": [
        "Forward AIC",
        "Backward AIC",
        "Stepwise AIC"
    ],
    "Selected Variables": [
        ", ".join(v for v in forward1_vars if v != "const"),
        ", ".join(v for v in backward1_vars if v != "const"),
        ", ".join(v for v in stepwise1_vars if v != "const")
    ],
    "Number of Predictors": [
        len([v for v in forward1_vars if v != "const"]),
        len([v for v in backward1_vars if v != "const"]),
        len([v for v in stepwise1_vars if v != "const"])
    ],
    "Final AIC": [
        round(forward1_model.aic, 2),
        round(backward1_model.aic, 2),
        round(stepwise1_model.aic, 2)
    ]
})

operational_results

,Method,Selected Variables,Number of Predictors,Final AIC
0,Forward AIC,"CSST, hurricane_force_diameter, lat, tropicals...",12,3305.0
1,Backward AIC,"year, day, lat, long, tropicalstorm_force_diam...",12,3305.0
2,Stepwise AIC,"CSST, hurricane_force_diameter, lat, tropicals...",12,3305.0


* AIC = 3305.00

NOTES:
* Forward Selection, Backward Elimination, and Stepwise Selection all converged on the same predictor set.
* This consistency suggests the selected variables provide a stable and robust model.
* The retained predictors include storm size measurements, geographic location, temporal variables, and several SHIPS environmental variables, indicating that both storm structure and environmental conditions contribute to distinguishing major hurricanes.

### Comparison table for Model 2

In [12]:
environmental_results = pd.DataFrame({
    "Method": [
        "Forward AIC",
        "Backward AIC",
        "Stepwise AIC"
    ],
    "Selected Variables": [
        ", ".join(forward2_vars),
        ", ".join(backward2_vars),
        ", ".join(stepwise2_vars)
    ],
    "AIC": [
        round(forward2_model.aic, 2),
        round(backward2_model.aic, 2),
        round(stepwise2_model.aic, 2)
    ]
})

environmental_results

,Method,Selected Variables,AIC
0,Forward AIC,"const, CSST, MTPW, long, COHC, day, RHMD, SHRD...",3831.78
1,Backward AIC,"const, year, month, day, long, CSST, COHC, SHR...",3831.78
2,Stepwise AIC,"const, CSST, MTPW, long, COHC, day, RHMD, SHRD...",3831.78


Model 2: Environmental Model

(Environmental, geographic, and temporal variables only)

* AIC = 3831.78

Notes:
* All three feature selection procedures converged on the same subset of predictors.
* The selected variables include sea surface temperature (CSST), ocean heat content (COHC), vertical wind shear (SHRD), relative humidity (RHMD), total precipitable water (MTPW), distance to land (DTL), and temporal and geographic variables.
* These findings indicate that environmental conditions alone provide meaningful predictive information for identifying major hurricanes, although the model does not fit as well as the Operational Model.